In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import h5py
import os
import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/sleep_research_io')
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import index_to_datetime_sleepdata, load_sleep_data, write_to_hdf5_file, read_in_routine, load_bm_data_aligned, merge_bm_and_airgo, airgo_resample_preprocess, bm_resample_interpolate
sys.path.append('C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code')
sys.path.append('/home/wolfgang/repos/Bedmaster-ICU-tools/code')
from research_bm_tools import BMR_load, get_metadata
from resample_BMR import remove_non_monotonic_data
from edw_functions import get_vitals, save_edw_vitals_df_for_each_mrn, get_edw_oxygen, get_vitals_single_mrn, get_edw_oxygen_single_mrn
from datetime import datetime, timedelta

In [3]:
# load mapping table as created in icu_sleep_data_mapping_table.ipynb

mapping_df = pd.read_csv('data_mapping_table_covid.csv')

## input
### bedmaster:
output from bedmaster_data_vitals_for_MRNs (maybe different in future), i.e. bedmaster .h5 file, default-name: MRN_{mrn}.h5
### airgo:
either raw airgo file [.csv] or a file with additional columns/signals (features, model outputs) [.csv or .h5 sleep_research_format]

### in general:
usually, patients are either identified via study_id or via mrn. read-in procedure shall be stable for these variants, i.e. what is needed is a dictionary that maps bedmaster file to airgo file / vice versa. this dictionary might need to be created in different ways for different projects, but from there onwards read in/processing should be standardized.

### params, settings:

In [3]:
do_resample_and_interpolation = False # IF only airgo exists, does it need to be resampled to 'perfect 10Hz'?

In [4]:
mapping_df.head(3)

,study_id,mrn,bm_file,airgo_file,bm_file_path,airgo_file_path,edw_vitals_file_path,edw_oxygen_supplement_file_path,output_file_path
0,1,6918597,MRN_6918597.h5,airgo_001.csv,/media/mad3/Projects/covid_data/h5_files/MRN_6...,/media/ssd2/Covid19_Respiration/Data_Analysis/...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/ssd2/Covid19_Respiration/Data_Analysis/...
1,2,6922857,MRN_6922857.h5,airgo_002.csv,/media/mad3/Projects/covid_data/h5_files/MRN_6...,/media/ssd2/Covid19_Respiration/Data_Analysis/...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/ssd2/Covid19_Respiration/Data_Analysis/...
2,3,6921854,MRN_6921854.h5,airgo_003.csv,/media/mad3/Projects/covid_data/h5_files/MRN_6...,/media/ssd2/Covid19_Respiration/Data_Analysis/...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/ssd2/Covid19_Respiration/Data_Analysis/...


In [ ]:
for irow, row in mapping_df.iterrows():
    
    try:
        
        airgo_path_tmp = row.airgo_file_path
        bm_path_tmp = row.bm_file_path
        output_path_tmp = row.output_file_path
        
        if os.path.exists(output_path_tmp):
            continue
            
        print(row.study_id)
        airgo_exists = os.path.exists(airgo_path_tmp)
        print(f'airgo_exists: {airgo_exists}')
        bm_exists = os.path.exists(bm_path_tmp)
        print(f'bm_exists: {bm_exists}')
        
        
        bm_exists = os.path.exists(bm_path_tmp)
        
        if airgo_exists:
            data_airgo, hdr_airgo, fs_airgo = read_in_routine(airgo_path_tmp)

        if bm_exists:
            # this load the bedmaster data that is saved in 'dataframe vitals' format, 
            # i.e. signals are already aligned and concatenated and posix is separate column. gets convertd to datetime
            data_bm = load_bm_data_aligned(bm_path_tmp)

        if airgo_exists & bm_exists:
            
#             print('tmp!!!!')
#             data_airgo = data_airgo.iloc[5000:5000+fs_airgo*60*120]
#             data_bm = data_bm.loc[data_airgo.index[0] : data_airgo.index[-1]]
            
            data_combined = merge_bm_and_airgo(data_bm, data_airgo)

        elif airgo_exists:
            if do_resample_and_interpolation:
                data_airgo = airgo_resample_preprocess(data_airgo)
            data_combined = data_airgo

        elif bm_exists:
            data_combined = bm_resample_interpolate(data_bm)

            
        ### ADD EDW:
        
        # vitals:
        edw_vitals = get_vitals_single_mrn(row.edw_vitals_file_path, row.mrn)
        edw_vitals.columns = ['edw_' + x for x in edw_vitals.columns]
        edw_vitals = edw_vitals.loc[data_combined.index[0] - timedelta(hours=1.5):data_combined.index[-1] + timedelta(hours=1.5)]
        edw_vitals = edw_vitals.asfreq('0.1S')
        data_combined = data_combined.join(edw_vitals, how='outer')
        
        # oxygen:
        edw_oxygen = get_edw_oxygen_single_mrn(row.edw_oxygen_supplement_file_path, row.mrn)
        edw_oxygen = edw_oxygen.loc[data_combined.index[0] - timedelta(hours=1.5):data_combined.index[-1] + timedelta(hours=1.5)]
        edw_oxygen = edw_oxygen.asfreq('0.1S')
        data_combined = data_combined.join(edw_oxygen, how='outer')
        
        
        hdr = {}
        dt = data_combined.index[0]
        hdr['start_datetime'] = np.array([dt.year, dt.month, dt.day,
                 dt.hour, dt.minute,
                 dt.second, dt.microsecond])
        hdr['MRN'] = row.mrn
        hdr['study_id'] = int(row.study_id)
        hdr['fs'] = 10

        write_to_hdf5_file(data_combined, output_path_tmp, hdr=hdr)
        
    except Exception as e:
        print(f'Exception for {row.study_id}:')
        print(e)
        continue
    

011
airgo_exists: True
bm_exists: True


In [99]:
# edw_oxygen = edw_oxygen.loc[data_combined.index[0] : data_combined.index[-1]]

In [150]:
data_combined.head(2)

,accx,accy,accz,band,band_unscaled,movavg_0_5s,movavg_1_2s,movavg_1min,deriv1,ventilation0,...,edw_bp_systolic,edw_pulse,edw_pulse_oximetry,edw_respirations,edw_temperature,edw_urine_output,oxygen_device,oxygen_flow_rate,positioning_frequency,repositioned
datetime,,,,,,,,,,,,,,,,,,,,,
2020-04-15 18:29:41.500,-0.137615,-0.566769,-0.790010,0.367826,2704.0,0.383283,0.376199,0.362775,0.018549,0.018549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-04-15 18:29:41.600,-0.131498,-0.568807,-0.792049,0.375555,2705.0,0.380192,0.373622,0.362852,-0.003091,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
mapping_df.loc[mapping_df.study_id=='015']


,study_id,mrn,bm_file,airgo_file,bm_file_path,airgo_file_path,edw_vitals_file_path,edw_oxygen_supplement_file_path,output_file_path
14,015,6969273,MRN_6969273.h5,airgo_015.csv,/media/mad3/Projects/covid_data/h5_files/MRN_6...,/media/ssd2/Covid19_Respiration/Data_Analysis/...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/ssd2/Covid19_Respiration/Data_Analysis/...


In [24]:
mapping_df.loc[[14]]

,study_id,mrn,bm_file,airgo_file,bm_file_path,airgo_file_path,edw_vitals_file_path,edw_oxygen_supplement_file_path,output_file_path
14,015,6969273,MRN_6969273.h5,airgo_015.csv,/media/mad3/Projects/covid_data/h5_files/MRN_6...,/media/ssd2/Covid19_Respiration/Data_Analysis/...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/mad3/Projects/covid_data/EDW/edw_airgo_...,/media/ssd2/Covid19_Respiration/Data_Analysis/...


In [22]:
output_w_predictions = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data_2'
files_ready = os.listdir(output_w_predictions)

In [24]:
# same as above except ONLY ADD OXYGEN EDW DATA TO DF.
# first add. 14 and 15
# then all
# then Combined_Data.


for irow, row in mapping_df.loc[[14]].iterrows():
    
    try:
        
        print(row.study_id)
        input_path_tmp = os.path.join(output_w_predictions, f'cov_resp_{row.study_id}.h5')
        output_path_tmp = input_path_tmp.replace('.h5', '_2.h5')
        
        data_combined, hdr, fs = read_in_routine(input_path_tmp)

        ### ADD EDW:
        
        # vitals:
#         edw_vitals = get_vitals_single_mrn(row.edw_vitals_file_path, row.mrn)
#         edw_vitals.columns = ['edw_' + x for x in edw_vitals.columns]
#         edw_vitals = edw_vitals.loc[data_combined.index[0] - timedelta(hours=1.5):data_combined.index[-1] + timedelta(hours=1.5)]
#         edw_vitals = edw_vitals.asfreq('0.1S')
#         data_combined = data_combined.join(edw_vitals, how='outer')
        
        # oxygen:
        edw_oxygen = get_edw_oxygen_single_mrn(row.edw_oxygen_supplement_file_path, row.mrn)
        edw_oxygen = edw_oxygen.loc[data_combined.index[0] - timedelta(hours=1.5):data_combined.index[-1] + timedelta(hours=1.5)]
        edw_oxygen = edw_oxygen.asfreq('0.1S')
        data_combined = data_combined.join(edw_oxygen, how='outer')
        
        hdr = {}
        dt = data_combined.index[0]
        hdr['start_datetime'] = np.array([dt.year, dt.month, dt.day,
                 dt.hour, dt.minute,
                 dt.second, dt.microsecond])
        hdr['MRN'] = row.mrn
        hdr['study_id'] = int(row.study_id)
        hdr['fs'] = 10

#         write_to_hdf5_file(data_combined, output_path_tmp, hdr=hdr, overwrite=True)
        
    except Exception as e:
        print(f'Exception for {row.study_id}:')
        print(e)
        continue
    

015


In [26]:
data_combined.columns[-1]

'repositioned'

In [44]:
data_combined.head(1)

,apnea_binary,apnea_end,acc_ai_10sec,acc_ai_1sec,acc_enmo,acc_enmo_10sec,acc_enmo_1sec,accx,accx_1sec,accy,...,ventilation_60s,ventilation_8s,ventilation_cvar_1min,ventilation_cvar_2min,ventilation_cvar_30sec,ventilation_cvar_5min,oxygen_device,oxygen_flow_rate,positioning_frequency,repositioned
2020-05-29 16:33:00,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None (Room air),NaN,NaN,NaN


In [45]:
data_combined.tail(1)

,apnea_binary,apnea_end,acc_ai_10sec,acc_ai_1sec,acc_enmo,acc_enmo_10sec,acc_enmo_1sec,accx,accx_1sec,accy,...,ventilation_60s,ventilation_8s,ventilation_cvar_1min,ventilation_cvar_2min,ventilation_cvar_30sec,ventilation_cvar_5min,oxygen_device,oxygen_flow_rate,positioning_frequency,repositioned
2020-06-07 11:48:04,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
edw_oxygen

,oxygen_device,oxygen_flow_rate,positioning_frequency,repositioned
datetime,,,,
2020-05-29 16:33:00.000,None (Room air),NaN,NaN,NaN
2020-05-29 16:33:00.100,NaN,NaN,NaN,NaN
2020-05-29 16:33:00.200,NaN,NaN,NaN,NaN
2020-05-29 16:33:00.300,NaN,NaN,NaN,NaN
2020-05-29 16:33:00.400,NaN,NaN,NaN,NaN
...,...,...,...,...
2020-06-07 09:47:59.600,NaN,NaN,NaN,NaN
2020-06-07 09:47:59.700,NaN,NaN,NaN,NaN
2020-06-07 09:47:59.800,NaN,NaN,NaN,NaN


In [47]:
'oxygen_device' in data_combined.columns

True

In [29]:
# save signals:
for signal_tmp in data_combined.columns:
    # print(signal_tmp)
    if signal_tmp in ['annotation', 'Annotation', 'oxygen_device', 'repositioned']: 
        print(signal_tmp)

oxygen_device
repositioned


In [34]:
# output_path_tmp = input_path_tmp
write_to_hdf5_file(data_combined, output_path_tmp, hdr=hdr, overwrite=True)


apnea_binary
apnea_end
acc_ai_10sec
acc_ai_1sec
acc_enmo
acc_enmo_10sec
acc_enmo_1sec
accx
accx_1sec
accy
accy_1sec
accz
accz_1sec
apnea_pred_ro_a3
apnea_pred_ro_a3_ss
apnea_pred_rsr_a3
apnea_pred_rsr_a3_ss
apnea_pred_va_a3
apnea_prob_ro_a3
apnea_prob_rsr_a3
band
band_unscaled
cuff
deriv1
edw_bp_diastolic
edw_bp_systolic
edw_pulse
edw_pulse_oximetry
edw_respirations
edw_temperature
edw_urine_output
envelope_lo
envelope_up
exht
hr
hypo_10_60
hypoxic_area
ibi
ibi_cvar_1min
ibi_cvar_2min
ibi_cvar_30sec
ibi_cvar_5min
ibi_mean_5min
ibi_std_5min
inht
inht_cycle_ratio
inht_cycle_ratio_10sec
inht_exht_ratio
inht_exht_ratio_10sec
instability_index_1min
instability_index_2min
instability_index_30sec
instability_index_5min
katz_fd_10s_smoothed
katz_fd_30s_smoothed
katz_fd_45s_smoothed
katz_fd_60s_smoothed
movavg_0_5s
movavg_0_5s_unscaled
movavg_1_2s
movavg_1min
movmedian_10min
movmedian_1min
movmedian_30sec
movmedian_5min
movstd_10min
movstd_10s
movstd_12s
movstd_15s
movstd_1min_unscaled
movstd_3

(Pdb)  c


oxygen_flow_rate
positioning_frequency
> /home/wolfgang/repos/sleep_research_io/sleep_research_functions.py(30)write_to_hdf5_file()
-> import pdb; pdb.set_trace()


(Pdb)  c


repositioned
> /home/wolfgang/repos/sleep_research_io/sleep_research_functions.py(31)write_to_hdf5_file()
-> dtype1 = h5py.string_dtype(encoding='utf-8') # h5py needs to be >= 2.10


(Pdb)  c


In [3]:
df = pd.read_csv(oxygen_supplement_file_path)

In [10]:
df[df.MRN == 6969273].head(2)

,MRN,PatientID,PatientEncounterID,InpatientDataID,DepartmentID,FlowsheetDataID,FlowsheetMeasureID,FlowsheetMeasureNM,RecordedDTS,MeasureTXT
10155,6969273,Z16445603,3.308541e+09,142529672,1.002001e+10,83365342,250026,R OXYGEN FLOW RATE,2020-05-29 17:00:00.0000000,3
10156,6969273,Z16445603,3.308541e+09,142529672,1.002001e+10,83365342,250026,R OXYGEN FLOW RATE,2020-05-29 18:10:00.0000000,3


In [4]:
edw_oxygen = get_edw_oxygen_single_mrn(oxygen_supplement_file_path, 6969273)

In [6]:
edw_oxygen.repositioned

datetime
2020-05-29 12:26:00    NaN
2020-05-29 12:39:00    NaN
2020-05-29 14:24:00    NaN
2020-05-29 16:33:00    NaN
2020-05-29 17:00:00    NaN
                      ... 
2020-06-06 17:22:00    NaN
2020-06-06 19:53:00    NaN
2020-06-07 05:10:00    NaN
2020-06-07 08:01:00    NaN
2020-06-07 09:48:00    NaN
Name: repositioned, Length: 122, dtype: object

In [10]:
# edw_oxygen = edw_oxygen.loc[data_combined.index[0] - timedelta(hours=1.5):data_combined.index[-1] + timedelta(hours=1.5)]
edw_oxygen = edw_oxygen.asfreq('0.1S')

In [12]:
edw_oxygen = edw_oxygen.astype('category')

In [16]:
edw_oxygen = edw_oxygen.astype(str)

In [11]:
import sys
sys.getsizeof(edw_oxygen)/1000

1044651.71

In [13]:
import sys
sys.getsizeof(edw_oxygen)/1000

92175.735

In [17]:
import sys
sys.getsizeof(edw_oxygen)/1000

1904939.782

In [19]:
edw_oxygen.columns[-1]

'repositioned'

In [15]:
edw_vitals = get_vitals_single_mrn(vitals_file_path, 6969273)

In [16]:
edw_vitals

,bp_diastolic,bp_systolic,pulse,pulse_oximetry,respirations,temperature,urine_output
datetime,,,,,,,
2020-05-29 12:26:00,57,107,98,92,18,98.8,NaN
2020-05-29 12:39:00,85,135,95,93,23,97.8,NaN
2020-05-29 14:24:00,74,125,86,94,18,98.1,NaN
2020-05-29 15:00:00,NaN,NaN,88,92,NaN,NaN,NaN
2020-05-29 16:33:00,79,133,88,89,26,100,NaN
...,...,...,...,...,...,...,...
2020-06-06 17:22:00,64,111,75,97,NaN,97.1,NaN
2020-06-06 19:53:00,71,113,66,94,18,97.7,NaN
2020-06-07 05:10:00,67,116,68,91,18,98.5,NaN
